In [1]:
print("hola")

hola


In [1]:
#!/usr/bin/env python3
"""
GDR Data Processor & Analyzer (Llama.cpp / Instructor Edition)
==============================================================

A robust post-processing script for Geothermal Data Repository datasets.
It performs the following:
1. Manages processing state via TinyDB.
2. Iterates through downloaded submissions (sorted by size).
3. Extracts text and statistical summaries from files (supports Archives, PDF, CSV, Excel).
4. VALIDATION: Checks if submission contains valid data files (.csv, .xlsx, .xls, .parquet),
   checking inside compressed files as well. If not, skips LLM analysis.
5. PROMPT LOGGING: Saves the exact prompt sent to the LLM in the result folder.
6. Uses a local Llama.cpp server (via Instructor) to evaluate data viability.
7. Saves results to JSON and Markdown in a dedicated subfolder.

Usage:
    # Ensure your llama.cpp server is running (e.g., ./server -m model.gguf --port 8000 --ctx-size 8192)
    export LLAMA_API_URL="http://localhost:8000/v1" 
    python processor.py
"""

import os

from dotenv import load_dotenv
load_dotenv()

import re
import json
import logging
import shutil
import zipfile
import tarfile
import tempfile
import traceback
from abc import ABC, abstractmethod
from pathlib import Path
from typing import List, Dict, Optional, Any

import pandas as pd
import pypdf
import instructor
from tinydb import TinyDB, Query
from openai import OpenAI
from pydantic import BaseModel, Field

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

# Paths
# BASE_DIR = Path(__file__).parent
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "gdr_data"
FILES_DIR = DATA_DIR / "files"
SCRAPER_DB_PATH = DATA_DIR / "gdr_db.json"
PROCESS_LOG_PATH = BASE_DIR / "processing_log.json"

# Settings
# Point this to your local llama.cpp server
LLAMA_API_URL = os.environ.get("LLAMA_API_URL", "http://localhost:8000/v1")
TARGET_MODEL = os.environ.get("TARGET_MODEL", "mistral-14b-instruct")
API_KEY = os.environ.get("LLAMA_API_KEY", "sk-no-key-required")

# Token / Context Limits
# Limit response generation tokens
MAX_OUTPUT_TOKENS = int(os.environ.get("MAX_OUTPUT_TOKENS", 4096))
# Limit input context characters (approx 4 chars/token). 
# 48000 chars ~= 12k tokens. Adjust based on your model's context window (e.g. 8k, 16k, 32k).
MAX_CONTEXT_CHARS = int(os.environ.get("MAX_CONTEXT_CHARS", 48000))

LOG_LEVEL = logging.INFO

# Logging Setup
logging.basicConfig(
    level=LOG_LEVEL,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("GDR_Processor")

# ------------------------------------------------------------------------------
# SYSTEM PROMPT
# ------------------------------------------------------------------------------

SYSTEM_PROMPT = """
You are a highly experienced Geothermal Reservoir Engineer and ORC Specialist.
Your primary task is to critically evaluate submitted datasets to determine their suitability for one of two specific applications:
1.  **Organic Rankine Cycle (ORC) Feasibility Study:** This specifically targets data relevant to utilizing **produced water from Oil & Gas operations** or **low-enthalpy geothermal resources** for power generation.
2.  **General Geothermal Reservoir Assessment:** This encompasses broader data needed for understanding and characterizing any type of geothermal resource.

**CRITICAL VIABILITY CRITERIA:**
A dataset is considered **VIABLE** ONLY IF it contains sufficiently detailed and relevant physical or chemical measurements that enable meaningful quantitative analysis or modeling for *either* of the above applications. This means the data must go beyond simple metadata or general descriptions and provide specific parameters.

**MANDATORY DATA REQUIREMENTS (MEET AT LEAST ONE OF THE FOLLOWING):**

*   **For ORC Feasibility (Oil/Gas Produced Water or Low-Enthalpy Geothermal):**
    *   **Set A (Thermodynamic & Flow):** Requires simultaneous measurement or reliable estimation of:
        *   **Fluid Temperature (T)** AND
        *   **Flow Rate (Q)** OR **Pressure (P)** of the working fluid (water, brine, steam).
    *   **Set B (Geochemistry & Thermodynamics):** Requires comprehensive geochemical analysis *along with* temperature data:
        *   **Key Geochemical Parameters:** Such as Total Dissolved Solids (TDS), pH, dissolved Silica (SiO2), alkalinity (carbonates), and dissolved gases (e.g., CO2, H2S).
        *   **AND** **Fluid Temperature (T)**.
        *   *Note:* This set is crucial for assessing scaling potential, corrosivity, and resource quality for low-temperature applications.

*   **For General Geothermal Reservoir Assessment:**
    *   **Set C (Reservoir Properties):** Requires data that describes the subsurface geological and thermal characteristics of the reservoir. This can include:
        *   **Temperature Gradients/Logs:** Measured at various depths.
        *   **Permeability/Porosity estimates or measurements.**
        *   **Lithological Data:** Descriptions of rock formations.
        *   **Hydrogeological Parameters:** Such as aquifer characteristics.
        *   **Direct measurements of Heat Flow.**
        *   **AND** **Fluid Chemistry (as per Set B)** or **Pressure/Flow data (as per Set A)** if available, as these are also vital for reservoir characterization.

**SOURCE CONTEXT:**
*   Data may originate from conventional **Geothermal Wells** or from **Oil & Gas fields** where produced water is being considered for co-production or reinjection with thermal recovery.

**OUTPUT INSTRUCTIONS:**
*   Thoroughly analyze the provided dataset description and extracted file contents.
*   **If `is_viable` is `False`:**
    *   Provide a precise and actionable `reasoning` explaining *why* the data is insufficient for either application. Clearly state which mandatory data points are missing or inadequate.
    *   Leave all optional fields (`file_descriptions`, `variable_dictionary`, `invenio_data`) as `null`.
*   **If `is_viable` is `True`:**
    *   You MUST accurately populate `file_descriptions`, `variable_dictionary`, and `invenio_data`.
    *   `file_descriptions`: Provide a brief, informative summary of the content of each key data file relevant to the analysis.
    *   `variable_dictionary`: List and briefly describe *all* identified physical and chemical variables found in the dataset that are relevant to either ORC feasibility or reservoir assessment.
    *   `invenio_data`: Generate a professional and academic metadata entry suitable for publication in a repository like InvenioRDM. This must include:
        *   A formal, descriptive `title` that accurately reflects the dataset's content and potential application.
        *   A `description` that elaborates on the dataset's origin, purpose, and key findings.
        *   A list of potential `creators` (if inferable, otherwise use placeholder like "Unknown").
        *   A comprehensive list of relevant scientific `keywords`.

**DO NOT:**
*   Fabricate data or analysis.
*   Make assumptions about data quality beyond what is presented.
*   Infer viability based on file names alone without content analysis.
"""

# ------------------------------------------------------------------------------
# PYDANTIC MODELS (STRUCTURED OUTPUT)
# ------------------------------------------------------------------------------

class InvenioMetadata(BaseModel):
    title: str = Field(..., description="Formal title for the dataset, accurately reflecting content and application.")
    description: str = Field(..., description="Comprehensive abstract/description of origin, purpose, and findings.")
    creators: List[Dict[str, str]] = Field(..., description="List of creators/authors (or 'Unknown').")
    keywords: List[str] = Field(..., description="Relevant scientific keywords (e.g., 'Enthalpy', 'Brine Chemistry').")

class GeothermalAnalysisResult(BaseModel):
    is_viable: bool = Field(..., description="True ONLY IF data meets Set A (Thermo/Flow), Set B (Geochem+T), or Set C (Reservoir Props) criteria.")
    reasoning: str = Field(..., description="Precise reasoning. If False, explain exactly which parameters (T, P, Q, Chemistry) are missing.")
    file_descriptions: Optional[Dict[str, str]] = Field(None, description="Summary of key data files relevant to analysis.")
    variable_dictionary: Optional[Dict[str, str]] = Field(None, description="Mapping of physical variables found (e.g., {'Temperature': 'Deg C', 'TDS': 'mg/L'}).")
    invenio_data: Optional[InvenioMetadata] = Field(None, description="Structured metadata for InvenioRDM (Title, Abstract, Authors).")

# ------------------------------------------------------------------------------
# STATE MANAGEMENT
# ------------------------------------------------------------------------------

class ProcessManager:
    """Manages the processing log using TinyDB."""
    def __init__(self, db_path):
        self.db = TinyDB(db_path)
        self.table = self.db.table('processing_status')
        self.Record = Query()

    def should_process(self, submission_id: str) -> bool:
        """Returns True if submission is not yet processed."""
        record = self.table.get(self.Record.id == submission_id)
        if not record:
            return True
        return record.get('status') not in ['processed', 'skipped']

    def mark_started(self, submission_id: str):
        self.table.upsert(
            {'id': submission_id, 'status': 'pending', 'timestamp': pd.Timestamp.now().isoformat()},
            self.Record.id == submission_id
        )

    def mark_completed(self, submission_id: str, viable: bool):
        self.table.update(
            {'status': 'processed', 'viable': viable},
            self.Record.id == submission_id
        )

    def mark_skipped(self, submission_id: str, reason: str):
        self.table.update(
            {'status': 'skipped', 'reason': reason},
            self.Record.id == submission_id
        )

# ------------------------------------------------------------------------------
# FILE HANDLERS (POLYMORPHIC EXTRACTION)
# ------------------------------------------------------------------------------

class FileHandler(ABC):
    def __init__(self, file_path: Path):
        self.file_path = file_path
        self.filename = file_path.name
        self.file_size = file_path.stat().st_size

    @abstractmethod
    def load_content(self) -> Any:
        pass

    @abstractmethod
    def generate_summary(self) -> str:
        pass

class TextHandler(FileHandler):
    """Handles PDF, TXT, MD, DOC, DOCX, PPTX files."""
    def load_content(self) -> str:
        ext = self.file_path.suffix.lower()
        text = ""
        try:
            if ext == '.pdf':
                with open(self.file_path, 'rb') as f:
                    reader = pypdf.PdfReader(f)
                    # Extract from first 5 and last 5 pages to save time/memory
                    pages = len(reader.pages)
                    page_indices = list(range(min(5, pages)))
                    if pages > 10:
                        page_indices += list(range(pages - 5, pages))
                    
                    for i in sorted(list(set(page_indices))):
                        page_text = reader.pages[i].extract_text()
                        if page_text:
                            text += page_text + "\n"
            elif ext in ['.docx', '.pptx']:
                from markitdown import MarkItDown
                md = MarkItDown(enable_plugins=False) # Set to True to enable plugins
                result = md.convert(str(self.file_path))
                text = result.text_content
            else:
                # Standard text files
                with open(self.file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text = f.read()
        except Exception as e:
            text = f"[Error reading text file: {e}]"
        return text

    def generate_summary(self) -> str:
        content = self.load_content()
        words = content.split()
        # Limit words per file to prevent blowing context budget
        limit = 3000
        if len(words) > limit:
            summary = " ".join(words[:limit//2]) + "\n...[Content Truncated]...\n" + " ".join(words[-(limit//2):])
        else:
            summary = " ".join(words)
        return f"File: {self.filename}\nType: Text Document\nContent Preview:\n{summary}\n"

class TabularHandler(FileHandler):
    """Handles CSV, Excel, Parquet."""
    
    def load_content(self) -> Dict[str, pd.DataFrame]:
        """
        Loads tabular data. 
        Returns a dictionary where keys are sheet names (or 'default') 
        and values are DataFrames.
        """
        ext = self.file_path.suffix.lower()
        dfs = {}
        try:
            if ext == '.csv':
                # Read sample
                dfs['default'] = pd.read_csv(self.file_path, nrows=500) 
            elif ext in ['.xls', '.xlsx']:
                # Read ALL sheets
                dfs = pd.read_excel(self.file_path, sheet_name=None, nrows=500)
            elif ext == '.parquet':
                dfs['default'] = pd.read_parquet(self.file_path)
        except Exception:
            return {} # Return empty dict on error
        return dfs

    def generate_summary(self) -> str:
        all_dfs = self.load_content()
        
        if not all_dfs:
            return f"File: {self.filename}\nType: Tabular\nNote: Could not parse or file is empty.\n"
        
        summary_parts = [f"File: {self.filename}\nType: Tabular Data"]

        # Iterate through every sheet found
        for sheet_name, df in all_dfs.items():
            summary_parts.append(f"\n--- Sheet: {sheet_name} ---")
            
            if df.empty:
                summary_parts.append("Sheet is empty.")
                continue

            cols = list(df.columns)
            summary_parts.append(f"Columns: {', '.join(map(str, cols))}")
            summary_parts.append(f"Shape (Sample): {df.shape}")

            # Check if there are any numerical columns
            numeric_cols = df.select_dtypes(include=['number'])
            
            if numeric_cols.empty:
                # Requirement: If no numerical variable, put the first 20 rows.
                # This helps identify issues with headers or file reading.
                summary_parts.append("Warning: No numerical variables found. Displaying first 20 rows of raw data for analysis:")
                # Using to_string for safety, helps LLM visualize structure
                raw_preview = df.head(20).to_string() 
                summary_parts.append(raw_preview)
            else:
                # Standard behavior: Statistical summary
                desc = df.describe().to_string()
                summary_parts.append(f"Statistical Summary:\n{desc}")
        
        return "\n".join(summary_parts) + "\n"

class BinaryHandler(FileHandler):
    """Handles Images and unknowns."""
    def load_content(self):
        return None

    def generate_summary(self) -> str:
        return f"File: {self.filename}\nType: Binary/Unknown\nSize: {self.file_size} bytes\n(Content not extracted)\n"

# ------------------------------------------------------------------------------
# SUBMISSION CONTEXT
# ------------------------------------------------------------------------------

class SubmissionContext:
    def __init__(self, folder_path: Path, scraper_metadata: Dict):
        self.folder_path = folder_path
        self.id = scraper_metadata.get('id')
        self.metadata = scraper_metadata
        self.file_handlers: List[FileHandler] = []
        self.temp_dir = None

    def _get_handler(self, file_path: Path) -> FileHandler:
        ext = file_path.suffix.lower()
        if ext in ['.csv', '.xlsx', '.xls', '.parquet']:
            return TabularHandler(file_path)
        elif ext in ['.pdf', '.txt', '.md', '.doc', '.docx', '.json']:
            return TextHandler(file_path)
        else:
            return BinaryHandler(file_path)

    def process_files(self):
        """Recursively scan directory, handle archives, instantiate handlers."""
        # 1. Handle Archives by extracting to a temp dir
        self.temp_dir = tempfile.mkdtemp(prefix=f"gdr_{self.id}_")
        
        # Strategy: Iterate folder. If archive, extract to temp. If normal, use as is.
        raw_files = [p for p in self.folder_path.rglob('*') if p.is_file()]
        
        for file_path in raw_files:
            if file_path.name == "citation.bib": continue # handled in metadata
            
            # Skip existing analysis results folder if re-running
            if "orc_analysis_results" in str(file_path): continue
            if "analysis_results" in str(file_path): continue

            ext = file_path.suffix.lower()
            if ext in ['.zip', '.tar', '.gz']:
                try:
                    if ext == '.zip':
                        with zipfile.ZipFile(file_path, 'r') as z:
                            z.extractall(self.temp_dir)
                    elif ext in ['.tar', '.gz']:
                        with tarfile.open(file_path, 'r') as t:
                            t.extractall(self.temp_dir)
                except Exception as e:
                    logger.warning(f"Failed to extract archive {file_path.name}: {e}")
                    # Treat as binary if extraction fails
                    self.file_handlers.append(BinaryHandler(file_path))
            else:
                self.file_handlers.append(self._get_handler(file_path))

        # Now scan extracted files in temp_dir
        for file_path in Path(self.temp_dir).rglob('*'):
            if file_path.is_file():
                # Avoid __MACOSX and hidden files
                if file_path.name.startswith('.') or '__MACOSX' in str(file_path):
                    continue
                self.file_handlers.append(self._get_handler(file_path))

    @property
    def has_tabular_data(self) -> bool:
        """Checks if any recognized tabular data files were found (including inside archives)."""
        return any(isinstance(h, TabularHandler) for h in self.file_handlers)

    def compile_context(self) -> str:
        """Aggregates metadata and file summaries for the LLM."""
        context = []
        context.append(f"--- DATASET METADATA ---\n")
        context.append(f"Title: {self.metadata.get('title')}")
        context.append(f"Description: {self.metadata.get('description')}")
        context.append(f"Keywords: {self.metadata.get('keywords')}")
        context.append(f"\n--- FILE CONTENTS ---\n")
        
        for handler in self.file_handlers:
            summary = handler.generate_summary()
            context.append(summary)
            context.append("-" * 40)
            
        return "\n".join(context)

    def cleanup(self):
        """Remove temp directory."""
        if self.temp_dir and os.path.exists(self.temp_dir):
            shutil.rmtree(self.temp_dir)

# ------------------------------------------------------------------------------
# LLM ANALYSIS
# ------------------------------------------------------------------------------

def analyze_submission(context_str: str, client: instructor.Instructor) -> GeothermalAnalysisResult:
    """Sends context to OpenAI/Llama.cpp via Instructor for structured analysis."""
    
    # SYSTEM_PROMPT is defined globally at the top of the file

    # Ensure context doesn't exceed model limits
    # Hard truncation logic
    truncated_context = context_str
    if len(context_str) > MAX_CONTEXT_CHARS:
        logger.warning(f"Context length ({len(context_str)} chars) exceeds limit. Truncating to {MAX_CONTEXT_CHARS}.")
        truncated_context = context_str[:MAX_CONTEXT_CHARS] + "\n...[Remaining Context Truncated due to Length]..."

    try:
        # Using instructor patched client
        resp = client.chat.completions.create(
            model=TARGET_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": truncated_context}
            ],
            response_model=GeothermalAnalysisResult#,
            #max_tokens=MAX_OUTPUT_TOKENS
        )
        return resp
    except Exception as e:
        logger.error(f"LLM Error: {e}")
        # Return a fail-safe negative result
        return GeothermalAnalysisResult(is_viable=False, reasoning=f"Analysis failed due to LLM error: {e}")

# ------------------------------------------------------------------------------
# MAIN LOGIC
# ------------------------------------------------------------------------------

def get_dir_size(path: Path) -> int:
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            total += p.stat().st_size
    return total

def load_scraper_db():
    if not SCRAPER_DB_PATH.exists():
        raise FileNotFoundError("Scraper database not found.")
    with open(SCRAPER_DB_PATH, 'r') as f:
        # TinyDB JSON format is {"_default": {"1": {...}, "2": {...}}}
        raw = json.load(f)
        return raw.get('_default', {})

def main():
    # 1. Setup Client for Llama.cpp via Instructor
    logger.info(f"Connecting to Local LLM at: {LLAMA_API_URL}")
    
    openai_client = OpenAI(
        base_url=LLAMA_API_URL,
        api_key=API_KEY 
    )
    
    # Initialize Instructor with JSON Schema Mode for Llama.cpp
    client = instructor.from_openai(
        client=openai_client,
        mode=instructor.Mode.JSON_SCHEMA
    )

    pm = ProcessManager(PROCESS_LOG_PATH)
    
    try:
        scraper_data = load_scraper_db()
    except FileNotFoundError:
        logger.warning("Scraper DB not found, proceeding without it.")
        scraper_data = {}

    # 2. Get Submissions and Sort by Size
    # submission_dirs maps ID -> Path
    submission_dirs = []
    
    if not FILES_DIR.exists():
        logger.error(f"Files directory {FILES_DIR} does not exist.")
        return

    logger.info("Scanning directories and calculating sizes...")
    for item in FILES_DIR.iterdir():
        if item.is_dir():
            # Format is usually "ID_Name" or just ID depending on scraper
            # We assume extraction of ID from the beginning of the folder name
            match = re.match(r'^(\d+)', item.name)
            if match:
                sub_id = match.group(1)
                size = get_dir_size(item)
                submission_dirs.append({'id': sub_id, 'path': item, 'size': size})

    # Sort small to large
    submission_dirs.sort(key=lambda x: x['size'])
    logger.info(f"Found {len(submission_dirs)} submissions. Starting processing.")

    # 3. Processing Loop
    for sub in submission_dirs:
        sub_id = sub['id']
        path = sub['path']

        # Check state
        if not pm.should_process(sub_id):
            continue

        logger.info(f"Processing ID: {sub_id} (Size: {sub['size']/1024:.2f} KB)")
        pm.mark_started(sub_id)

        # Find original metadata from scraper DB
        original_meta = {}
        for key, val in scraper_data.items():
            if str(val.get('id')) == str(sub_id):
                original_meta = val
                break
        
        if not original_meta:
            logger.warning(f"Metadata missing for ID {sub_id}. Using directory info only.")
            original_meta = {'id': sub_id, 'title': path.name, 'description': '', 'keywords': []}

        # Context Creation
        ctx = SubmissionContext(path, original_meta)
        try:
            # 1. Process files (Extract zips, identify types)
            ctx.process_files()
            
            # 2. CHECK VIABILITY: Data File Existence
            if not ctx.has_tabular_data:
                reason = "No viable tabular data files (.csv, .xlsx, .xls, .parquet) found in submission."
                logger.info(f"  [SKIPPED] {reason}")
                pm.mark_skipped(sub_id, reason=reason)
                continue

            # 3. Compile Context
            context_str = ctx.compile_context()
            
            # Create a dedicated subfolder for results immediately
            results_dir = path / "analysis_results"
            results_dir.mkdir(exist_ok=True)

            # 4. SAVE PROMPT
            full_prompt_log = f"--- SYSTEM PROMPT ---\n{SYSTEM_PROMPT}\n\n--- USER CONTEXT ---\n{context_str}"
            prompt_file = results_dir / "llm_prompt.txt"
            with open(prompt_file, "w", encoding="utf-8") as f:
                f.write(full_prompt_log)
            logger.info(f"  Prompt saved to {prompt_file.name}")
            
            # 5. LLM Analysis
            logger.info("  Sending to LLM...")
            result = analyze_submission(context_str, client)
            
            # Save the raw classification result
            with open(results_dir / "classification_raw.json", "w") as f:
                json.dump(result.model_dump(), f, indent=2)

            if result.is_viable:
                logger.info("  [VIABLE] Generating Output Artifacts...")
                
                # Create Merged Metadata JSON
                final_metadata = original_meta.copy()
                final_metadata['analysis'] = {
                    'is_viable': True,
                    'reasoning': result.reasoning,
                    'variable_dictionary': result.variable_dictionary
                }
                # Ensure invenio_draft is populated
                if result.invenio_data:
                    final_metadata['invenio_draft'] = result.invenio_data.model_dump()
                else:
                    final_metadata['invenio_draft'] = {}

                final_metadata['file_descriptions'] = result.file_descriptions

                # Save to results_dir
                with open(results_dir / "metadata_analysis.json", "w") as f:
                    json.dump(final_metadata, f, indent=2)

                # Create Description Markdown
                invenio_title = result.invenio_data.title if result.invenio_data else original_meta.get('title', 'Untitled')
                invenio_desc = result.invenio_data.description if result.invenio_data else "No description provided."

                md_content = f"""# {invenio_title}

**Original ID:** {sub_id}  
**Status:** Viable for Geothermal/ORC Analysis

## Description
{invenio_desc}

## Variables Present
{json.dumps(result.variable_dictionary, indent=2) if result.variable_dictionary else "None listed"}

## Analysis Reasoning
{result.reasoning}
"""
                with open(results_dir / "description.md", "w") as f:
                    f.write(md_content)

                pm.mark_completed(sub_id, viable=True)

            else:
                logger.info(f"  [NOT VIABLE] {result.reasoning[:100]}...")
                pm.mark_skipped(sub_id, reason=result.reasoning)

        except Exception as e:
            logger.error(f"  Error processing {sub_id}: {e}")
            traceback.print_exc()
            pm.mark_skipped(sub_id, reason=f"Script Error: {str(e)}")
        finally:
            ctx.cleanup()

    logger.info("Processing complete.")

if __name__ == "__main__":
    main()

14:02:31 [INFO] Connecting to Local LLM at: http://intel1:8033/v1
14:02:31 [INFO] Scanning directories and calculating sizes...
14:02:31 [INFO] Found 179 submissions. Starting processing.
14:02:33 [INFO] Processing complete.
